# 04 — Interpretability & XAI
Using Captum's Occlusion method to explain model decisions on individual images.

In [ ]:
import sys; sys.path.insert(0, '..')
import warnings; warnings.filterwarnings('ignore')
import numpy as np
import torch
import joblib
import matplotlib.pyplot as plt
import matplotlib.cm as cm
from PIL import Image
from torchvision import transforms
from pathlib import Path

from src.config import paths, preprocessing_config, model_config
from src.embeddings import get_feature_extractor
from src.interpretability import generate_occlusion_heatmap

ROOT = Path('..')
device = torch.device('cpu')

print("Loading ResNet50 feature extractor...")
feature_extractor, _ = get_feature_extractor('resnet50')

print("Loading SVM classifier...")
classifier = joblib.load(ROOT / 'models' / 'best_classifier.joblib')
print("Models ready.")

## Select & Preprocess a Sample Image

In [ ]:
import pandas as pd, random
df = pd.read_csv(ROOT / 'data' / 'metadata.csv')

# Pick one benign and one malignant sample
benign_row = df[df['label']==0].sample(1, random_state=7).iloc[0]
malig_row  = df[df['label']==1].sample(1, random_state=7).iloc[0]

transform = transforms.Compose([
    transforms.Resize(preprocessing_config.IMAGE_SIZE),
    transforms.ToTensor(),
    transforms.Normalize(mean=preprocessing_config.NORMALIZE_MEAN,
                         std=preprocessing_config.NORMALIZE_STD),
])

def load_tensor(row):
    img_path = ROOT / 'data' / (row['image_id'] + '.jpg')
    return transform(Image.open(img_path).convert('RGB')), row['label']

benign_tensor, _ = load_tensor(benign_row)
malig_tensor,  _ = load_tensor(malig_row)

def denorm(t):
    mean = torch.tensor(preprocessing_config.NORMALIZE_MEAN).view(3,1,1)
    std  = torch.tensor(preprocessing_config.NORMALIZE_STD).view(3,1,1)
    return ((t * std + mean).clamp(0,1).permute(1,2,0).numpy() * 255).astype('uint8')

fig, axes = plt.subplots(1, 2, figsize=(8, 4))
axes[0].imshow(denorm(benign_tensor)); axes[0].set_title('Benign Sample', color='green', fontweight='bold'); axes[0].axis('off')
axes[1].imshow(denorm(malig_tensor)); axes[1].set_title('Malignant Sample', color='red', fontweight='bold'); axes[1].axis('off')
plt.suptitle('Selected Test Images', fontsize=13); plt.tight_layout(); plt.show()

## Generate XAI Occlusion Heatmaps

**How Occlusion works:** A patch of pixels is systematically covered. If hiding a region causes the model's confidence to drop, that region is deemed *important* (red). If covering it has no effect, it's *unimportant* (green).

In [ ]:
def get_prediction(tensor):
    with torch.no_grad():
        feat = feature_extractor(tensor.unsqueeze(0).to(device)).cpu().numpy()
    pred = classifier.predict(feat)[0]
    prob = classifier.predict_proba(feat)[0]
    return pred, prob

def make_heatmap_panels(tensor, label_name, target_class):
    print(f"Generating heatmap for {label_name} (target_class={target_class})...")
    attr = generate_occlusion_heatmap(tensor, feature_extractor, classifier, target_class)
    orig = denorm(tensor)
    attr_np = attr.permute(1,2,0).numpy()
    heat = np.abs(attr_np).sum(axis=-1)
    if heat.max() > 0: heat = (heat - heat.min()) / (heat.max() - heat.min())
    blend = (0.5 * (cm.RdYlGn_r(heat)[:,:,:3] * 255) + 0.5 * orig).astype('uint8')
    return orig, heat, blend

pred_b, prob_b = get_prediction(benign_tensor)
pred_m, prob_m = get_prediction(malig_tensor)
print(f"Benign sample   -> Predicted: {'Benign' if pred_b==0 else 'Malignant'} (confidence {max(prob_b)*100:.1f}%)")
print(f"Malignant sample -> Predicted: {'Benign' if pred_m==0 else 'Malignant'} (confidence {max(prob_m)*100:.1f}%)")

In [ ]:
orig_b, heat_b, blend_b = make_heatmap_panels(benign_tensor, 'Benign', int(pred_b))
orig_m, heat_m, blend_m = make_heatmap_panels(malig_tensor, 'Malignant', int(pred_m))

fig, axes = plt.subplots(2, 3, figsize=(14, 9))
rows = [(orig_b, heat_b, blend_b, 'Benign'), (orig_m, heat_m, blend_m, 'Malignant')]
cols = ['Original Image', 'Attribution Map', 'Overlay (Important Regions)']

for row_idx, (orig, heat, blend, lbl) in enumerate(rows):
    color = 'green' if lbl == 'Benign' else 'red'
    axes[row_idx][0].imshow(orig)
    axes[row_idx][1].imshow(heat, cmap='RdYlGn_r', vmin=0, vmax=1)
    axes[row_idx][2].imshow(blend)
    for col_idx in range(3):
        axes[row_idx][col_idx].axis('off')
        if col_idx == 0:
            axes[row_idx][col_idx].set_title(f'{lbl} — {cols[col_idx]}', color=color, fontweight='bold')
        else:
            axes[row_idx][col_idx].set_title(cols[col_idx], fontsize=10)

plt.suptitle('Captum Occlusion XAI Heatmaps — Red = High Model Attention',
             fontsize=13, fontweight='bold')
plt.tight_layout(); plt.show()
print("Red regions = pixels that, when covered, most reduce prediction confidence.")
print("These are the lesion features the SVM's learned embedding space focuses on.")